<a href="https://colab.research.google.com/github/wooriapt79/mulberry-/blob/main/mulberry_wifi_sensing_mvp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import random
import time
from dataclasses import dataclass
from typing import Dict, List


@dataclass
class DetectionResult:
    label: str
    confidence: float


class CSIReader:
    def __init__(self, interface: str = "wlan0") -> None:
        self.interface = interface

    def read_csi(self) -> List[float]:
        return [random.random() for _ in range(64)]


class CSIAnalyzer:
    def collect_for_duration(self, seconds: int) -> List[List[float]]:
        frames = max(1, min(seconds, 10))
        return [[random.random() for _ in range(64)] for _ in range(frames)]


class CSICollector:
    def collect(self, duration: int = 3) -> List[List[float]]:
        frames = max(1, duration * 5)
        return [[random.random() for _ in range(64)] for _ in range(frames)]


class LEDIndicator:
    def blink_green(self) -> None:
        print("[LED] blinking green")

    def off(self) -> None:
        print("[LED] off")


class MHCMotionDetector:
    def analyze(self, csi_data: List[float]) -> str:
        fall_threshold = 0.85  # 변경된 임계값
        abnormal_threshold = 0.60  # 변경된 임계값

        avg = sum(csi_data) / len(csi_data)
        if avg > fall_threshold:
            return "fall_detected"
        if avg > abnormal_threshold:
            return "abnormal_movement"
        return "normal"


class MHCMotionModel:
    def predict(self, amplitude_data: List[float]) -> DetectionResult:
        avg = sum(amplitude_data) / len(amplitude_data)
        if avg > 0.85:
            return DetectionResult(label="fall_detected", confidence=0.97)
        if avg > 0.68:
            return DetectionResult(label="abnormal_movement", confidence=0.83)
        return DetectionResult(label="normal", confidence=0.60)


class MHCMultiModalModel:
    def predict(self, aligned_data: Dict[str, List[float]]) -> str:
        audio = aligned_data.get("audio", [])
        csi = aligned_data.get("csi", [])
        signal = 0.0
        if audio:
            signal += sum(audio) / len(audio)
        if csi:
            signal += sum(csi) / len(csi)
        signal /= 2 if audio and csi else 1
        if signal > 0.82:
            return "fall_detected"
        if signal > 0.66:
            return "abnormal_movement"
        return "normal"


def load_mhc_model() -> MHCMultiModalModel:
    return MHCMultiModalModel()


class WiFiSensingModule:
    def __init__(self, interface: str = "wlan0") -> None:
        self.csi_extractor = CSIReader(interface)
        self.motion_detector = MHCMotionDetector()

    def alert_emergency(self) -> None:
        print("[EMERGENCY] Fall detected. Alert triggered.")

    def run_once(self) -> str:
        csi_data = self.csi_extractor.read_csi()
        motion_status = self.motion_detector.analyze(csi_data)
        print(f"[WiFiSensingModule] motion_status={motion_status}")
        if motion_status == "fall_detected":
            self.alert_emergency()
        return motion_status

    def run(self, iterations: int = 5, sleep_sec: float = 0.2) -> None:
        for _ in range(iterations):
            self.run_once()
            time.sleep(sleep_sec)


class WiFiSensingPrivacy:
    def __init__(self) -> None:
        self.user_consent = False
        self.collection_active = False
        self.led = LEDIndicator()

    def tts_speak(self, message: str) -> None:
        print(f"[TTS] {message}")

    def request_consent(self) -> None:
        self.tts_speak("움직임 감지 기능을 켤까요? 언제든지 끌 수 있어요.")
        self.user_consent = True
        print("[Privacy] user_consent=True")

    def toggle_collection(self, voice_command: str) -> None:
        if "움직임 감지 켜줘" in voice_command:
            self.collection_active = True
            self.led.blink_green()
            print("[Privacy] collection_active=True")
        elif "움직임 감지 꺼줘" in voice_command:
            self.collection_active = False
            self.led.off()
            print("[Privacy] collection_active=False")


class MHCMultiModalSensor:
    def __init__(self) -> None:
        self.mhc_model = load_mhc_model()

    def tts_speak(self, message: str) -> None:
        print(f"[TTS] {message}")

    def trigger_emergency(self) -> None:
        print("[EMERGENCY] Multimodal emergency trigger")

    def is_speech_present(self, audio_stream: List[float]) -> bool:
        return bool(audio_stream) and (sum(audio_stream) / len(audio_stream)) > 0.20

    def align_modalities(self, audio_stream: List[float], csi_stream: List[float]) -> Dict[str, List[float]]:
        return {"audio": audio_stream, "csi": csi_stream}

    def analyze(self, audio_stream: List[float], csi_stream: List[float]) -> str:
        aligned_data = self.align_modalities(audio_stream, csi_stream)
        situation = self.mhc_model.predict(aligned_data)

        if situation == "fall_detected" and self.is_speech_present(audio_stream):
            self.trigger_emergency()
        elif situation == "abnormal_movement":
            self.tts_speak("괜찮으세요?")

        print(f"[MHCMultiModalSensor] situation={situation}")
        return situation


class WhoFiIdentifier:
    def __init__(self) -> None:
        self.csi_analyzer = CSIAnalyzer()
        self.body_signature_db: Dict[str, List[float]] = {}

    def extract_biometric_features(self, csi_patterns: List[List[float]]) -> List[float]:
        if not csi_patterns:
            return [0.0] * 64
        cols = len(csi_patterns[0])
        return [
            sum(frame[i] for frame in csi_patterns) / len(csi_patterns)
            for i in range(cols)
        ]

    def compare_patterns(self, realtime_csi: List[float], signature: List[float]) -> float:
        if not realtime_csi or not signature or len(realtime_csi) != len(signature):
            return 0.0
        distance = sum(abs(a - b) for a, b in zip(realtime_csi, signature)) / len(signature)
        return max(0.0, 1.0 - distance)

    def enroll_person(self, person_id: str, seconds: int = 5) -> None:
        csi_patterns = self.csi_analyzer.collect_for_duration(seconds)
        signature = self.extract_biometric_features(csi_patterns)
        self.body_signature_db[person_id] = signature
        print(f"[WhoFi] enrolled person_id={person_id}")

    def identify_person(self, realtime_csi: List[float], threshold: float = 0.95) -> str:
        for person_id, signature in self.body_signature_db.items():
            similarity = self.compare_patterns(realtime_csi, signature)
            print(f"[WhoFi] compare {person_id}: similarity={similarity:.3f}")
            if similarity > threshold:
                return person_id
        return "unknown"


class WiFiFallDetector:
    def __init__(self) -> None:
        self.csi_collector = CSICollector()
        self.motion_analyzer = MHCMotionModel()

    def trigger_emergency(self) -> None:
        print("[EMERGENCY] WiFi fall detector triggered")

    def extract_amplitude(self, csi_stream: List[List[float]]) -> List[float]:
        return [sum(frame) / len(frame) for frame in csi_stream if frame]

    def detect_fall(self) -> DetectionResult:
        csi_stream = self.csi_collector.collect(duration=3)
        amplitude_data = self.extract_amplitude(csi_stream)
        result = self.motion_analyzer.predict(amplitude_data)
        print(f"[WiFiFallDetector] result={result.label}, confidence={result.confidence:.2f}")
        if result.label == "fall_detected" and result.confidence > 0.95:
            self.trigger_emergency()
        return result


def demo() -> None:
    print("\n=== 1. Privacy Control ===")
    privacy = WiFiSensingPrivacy()
    privacy.request_consent()
    privacy.toggle_collection("움직임 감지 켜줘")

    print("\n=== 2. WiFi Sensing Module ===")
    sensing = WiFiSensingModule()
    sensing.run(iterations=3, sleep_sec=0.1)

    print("\n=== 3. Multimodal Sensor ===")
    multimodal = MHCMultiModalSensor()
    audio_stream = [random.random() for _ in range(32)]
    csi_stream = [random.random() for _ in range(64)]
    multimodal.analyze(audio_stream, csi_stream)

    print("\n=== 4. WhoFi Identifier ===")
    identifier = WhoFiIdentifier()
    identifier.enroll_person("senior_01")
    sample = identifier.body_signature_db["senior_01"][:]
    identified = identifier.identify_person(sample, threshold=0.80)
    print(f"[WhoFi] identified={identified}")

    print("\n=== 5. Fall Detector ===")
    fall_detector = WiFiFallDetector()
    fall_detector.detect_fall()


if __name__ == "__main__":
    demo()



=== 1. Privacy Control ===
[TTS] 움직임 감지 기능을 켤까요? 언제든지 끌 수 있어요.
[Privacy] user_consent=True
[LED] blinking green
[Privacy] collection_active=True

=== 2. WiFi Sensing Module ===
[WiFiSensingModule] motion_status=normal
[WiFiSensingModule] motion_status=normal
[WiFiSensingModule] motion_status=normal

=== 3. Multimodal Sensor ===
[MHCMultiModalSensor] situation=normal

=== 4. WhoFi Identifier ===
[WhoFi] enrolled person_id=senior_01
[WhoFi] compare senior_01: similarity=1.000
[WhoFi] identified=senior_01

=== 5. Fall Detector ===
[WiFiFallDetector] result=normal, confidence=0.60


# Mulberry WiFi Sensing MVP (Minimum Viable Product)

이 프로젝트는 WiFi 신호(CSI: Channel State Information)를 활용하여 사람의 움직임 감지, 낙상 감지, 멀티모달 센싱, 그리고 사람 식별 기능을 시뮬레이션하는 MVP(Minimum Viable Product) 버전입니다.

## 주요 기능 및 클래스

*   **`CSIReader`**: WiFi 인터페이스로부터 CSI 데이터를 읽는 것을 모의합니다.
*   **`CSIAnalyzer`**: CSI 데이터를 수집하고 분석하는 기본적인 기능을 제공합니다.
*   **`CSICollector`**: 지정된 시간 동안 CSI 데이터를 수집하는 것을 모의합니다.
*   **`LEDIndicator`**: 시스템 상태를 시각적으로 나타내는 LED 동작을 모의합니다.
*   **`MHCMotionDetector`**: CSI 데이터의 평균 진폭을 기반으로 `normal`, `abnormal_movement`, `fall_detected`를 분석합니다. 임계값(`fall_threshold`, `abnormal_threshold`)을 조정할 수 있습니다.
*   **`MHCMotionModel`**: 진폭 데이터를 사용하여 `normal`, `abnormal_movement`, `fall_detected`를 예측하고 신뢰도를 반환합니다.
*   **`MHCMultiModalModel`**: 오디오와 CSI 데이터를 결합한 신호의 평균을 기반으로 상황을 예측합니다.
*   **`WiFiSensingModule`**: CSI 추출 및 `MHCMotionDetector`를 활용하여 움직임을 감지하고, 낙상 시 비상 알림을 트리거합니다.
*   **`WiFiSensingPrivacy`**: 사용자 동의 및 데이터 수집 활성화/비활성화를 관리하며, 음성 명령에 반응하는 것을 모의합니다.
*   **`MHCMultiModalSensor`**: `MHCMultiModalModel`을 사용하여 오디오와 CSI 데이터를 통합 분석하고, 음성 존재 여부, 낙상 및 비정상 움직임 감지 시 특정 액션을 수행합니다.
*   **`WhoFiIdentifier`**: CSI 패턴을 '바디 시그니처'로 사용하여 사람을 등록하고 식별합니다.
*   **`WiFiFallDetector`**: `CSICollector`와 `MHCMotionModel`을 사용하여 낙상을 감지하고, 높은 신뢰도로 낙상 감지 시 비상 알림을 트리거합니다.

## 프로젝트 실행 방법

Google Colab 환경에서 이 노트북(`mulberry_wifi_sensing_mvp.ipynb`와 같은 파일)을 엽니다. 모든 셀을 순서대로 실행하여 각 모듈의 동작을 확인할 수 있습니다.

*   **모든 셀 실행**: Colab 메뉴에서 `런타임(Runtime)` -> `모두 실행(Run all)`을 선택합니다.
*   **개별 셀 실행**: 원하는 셀을 선택한 후 `Ctrl + Enter` 또는 셀 좌측의 실행 버튼(▶)을 클릭합니다.

## 시뮬레이션 및 테스트

노트북에는 각 클래스 및 모듈의 기능을 테스트하기 위한 다양한 시뮬레이션 코드가 포함되어 있습니다. 이를 통해 각 모듈의 동작 방식과 임계값 변경에 따른 영향을 확인할 수 있습니다.

### `MHCMotionDetector` 임계값 조정
`MHCMotionDetector` 클래스 내 `fall_threshold` 및 `abnormal_threshold` 변수의 값을 변경하여 감지 민감도를 조정할 수 있습니다.

### `WiFiFallDetector` 낙상 시뮬레이션
`MockCSICollector`를 통해 고의적으로 낙상 상황과 유사한 CSI 데이터를 주입하여 비상 알림이 트리거되는 과정을 확인할 수 있습니다.

### `WhoFiIdentifier` 사람 식별
특정 `person_id`로 사람의 바디 시그니처를 등록한 후, 유사하거나 다른 CSI 데이터로 식별을 시도하여 모듈의 정확도를 테스트할 수 있습니다.

# Mulberry WiFi Sensing MVP (Minimum Viable Product)

이 프로젝트는 WiFi 신호(CSI: Channel State Information)를 활용하여 사람의 움직임 감지, 낙상 감지, 멀티모달 센싱, 그리고 사람 식별 기능을 시뮬레이션하는 MVP(Minimum Viable Product) 버전입니다.

## 주요 기능 및 클래스

*   **`CSIReader`**: WiFi 인터페이스로부터 CSI 데이터를 읽는 것을 모의합니다.
*   **`CSIAnalyzer`**: CSI 데이터를 수집하고 분석하는 기본적인 기능을 제공합니다.
*   **`CSICollector`**: 지정된 시간 동안 CSI 데이터를 수집하는 것을 모의합니다.
*   **`LEDIndicator`**: 시스템 상태를 시각적으로 나타내는 LED 동작을 모의합니다.
*   **`MHCMotionDetector`**: CSI 데이터의 평균 진폭을 기반으로 `normal`, `abnormal_movement`, `fall_detected`를 분석합니다. 임계값(`fall_threshold`, `abnormal_threshold`)을 조정할 수 있습니다.
*   **`MHCMotionModel`**: 진폭 데이터를 사용하여 `normal`, `abnormal_movement`, `fall_detected`를 예측하고 신뢰도를 반환합니다.
*   **`MHCMultiModalModel`**: 오디오와 CSI 데이터를 결합한 신호의 평균을 기반으로 상황을 예측합니다.
*   **`WiFiSensingModule`**: CSI 추출 및 `MHCMotionDetector`를 활용하여 움직임을 감지하고, 낙상 시 비상 알림을 트리거합니다.
*   **`WiFiSensingPrivacy`**: 사용자 동의 및 데이터 수집 활성화/비활성화를 관리하며, 음성 명령에 반응하는 것을 모의합니다.
*   **`MHCMultiModalSensor`**: `MHCMultiModalModel`을 사용하여 오디오와 CSI 데이터를 통합 분석하고, 음성 존재 여부, 낙상 및 비정상 움직임 감지 시 특정 액션을 수행합니다.
*   **`WhoFiIdentifier`**: CSI 패턴을 '바디 시그니처'로 사용하여 사람을 등록하고 식별합니다.
*   **`WiFiFallDetector`**: `CSICollector`와 `MHCMotionModel`을 사용하여 낙상을 감지하고, 높은 신뢰도로 낙상 감지 시 비상 알림을 트리거합니다.

## 프로젝트 실행 방법

Google Colab 환경에서 이 노트북(`mulberry_wifi_sensing_mvp.ipynb`와 같은 파일)을 엽니다. 모든 셀을 순서대로 실행하여 각 모듈의 동작을 확인할 수 있습니다.

*   **모든 셀 실행**: Colab 메뉴에서 `런타임(Runtime)` -> `모두 실행(Run all)`을 선택합니다.
*   **개별 셀 실행**: 원하는 셀을 선택한 후 `Ctrl + Enter` 또는 셀 좌측의 실행 버튼(▶)을 클릭합니다.

## 시뮬레이션 및 테스트

노트북에는 각 클래스 및 모듈의 기능을 테스트하기 위한 다양한 시뮬레이션 코드가 포함되어 있습니다. 이를 통해 각 모듈의 동작 방식과 임계값 변경에 따른 영향을 확인할 수 있습니다.

### `MHCMotionDetector` 임계값 조정
`MHCMotionDetector` 클래스 내 `fall_threshold` 및 `abnormal_threshold` 변수의 값을 변경하여 감지 민감도를 조정할 수 있습니다.

### `WiFiFallDetector` 낙상 시뮬레이션
`MockCSICollector`를 통해 고의적으로 낙상 상황과 유사한 CSI 데이터를 주입하여 비상 알림이 트리거되는 과정을 확인할 수 있습니다.

### `WhoFiIdentifier` 사람 식별
특정 `person_id`로 사람의 바디 시그니처를 등록한 후, 유사하거나 다른 CSI 데이터로 식별을 시도하여 모듈의 정확도를 테스트할 수 있습니다.

# Mulberry WiFi Sensing MVP (Minimum Viable Product)

이 프로젝트는 WiFi 신호(CSI: Channel State Information)를 활용하여 사람의 움직임 감지, 낙상 감지, 멀티모달 센싱, 그리고 사람 식별 기능을 시뮬레이션하는 MVP(Minimum Viable Product) 버전입니다.

## 주요 기능 및 클래스

*   **`CSIReader`**: WiFi 인터페이스로부터 CSI 데이터를 읽는 것을 모의합니다.
*   **`CSIAnalyzer`**: CSI 데이터를 수집하고 분석하는 기본적인 기능을 제공합니다.
*   **`CSICollector`**: 지정된 시간 동안 CSI 데이터를 수집하는 것을 모의합니다.
*   **`LEDIndicator`**: 시스템 상태를 시각적으로 나타내는 LED 동작을 모의합니다.
*   **`MHCMotionDetector`**: CSI 데이터의 평균 진폭을 기반으로 `normal`, `abnormal_movement`, `fall_detected`를 분석합니다. 임계값(`fall_threshold`, `abnormal_threshold`)을 조정할 수 있습니다.
*   **`MHCMotionModel`**: 진폭 데이터를 사용하여 `normal`, `abnormal_movement`, `fall_detected`를 예측하고 신뢰도를 반환합니다.
*   **`MHCMultiModalModel`**: 오디오와 CSI 데이터를 결합한 신호의 평균을 기반으로 상황을 예측합니다.
*   **`WiFiSensingModule`**: CSI 추출 및 `MHCMotionDetector`를 활용하여 움직임을 감지하고, 낙상 시 비상 알림을 트리거합니다.
*   **`WiFiSensingPrivacy`**: 사용자 동의 및 데이터 수집 활성화/비활성화를 관리하며, 음성 명령에 반응하는 것을 모의합니다.
*   **`MHCMultiModalSensor`**: `MHCMultiModalModel`을 사용하여 오디오와 CSI 데이터를 통합 분석하고, 음성 존재 여부, 낙상 및 비정상 움직임 감지 시 특정 액션을 수행합니다.
*   **`WhoFiIdentifier`**: CSI 패턴을 '바디 시그니처'로 사용하여 사람을 등록하고 식별합니다.
*   **`WiFiFallDetector`**: `CSICollector`와 `MHCMotionModel`을 사용하여 낙상을 감지하고, 높은 신뢰도로 낙상 감지 시 비상 알림을 트리거합니다.

## 프로젝트 실행 방법

Google Colab 환경에서 이 노트북(`mulberry_wifi_sensing_mvp.ipynb`와 같은 파일)을 엽니다. 모든 셀을 순서대로 실행하여 각 모듈의 동작을 확인할 수 있습니다.

*   **모든 셀 실행**: Colab 메뉴에서 `런타임(Runtime)` -> `모두 실행(Run all)`을 선택합니다.
*   **개별 셀 실행**: 원하는 셀을 선택한 후 `Ctrl + Enter` 또는 셀 좌측의 실행 버튼(▶)을 클릭합니다.

## 시뮬레이션 및 테스트

노트북에는 각 클래스 및 모듈의 기능을 테스트하기 위한 다양한 시뮬레이션 코드가 포함되어 있습니다. 이를 통해 각 모듈의 동작 방식과 임계값 변경에 따른 영향을 확인할 수 있습니다.

### `MHCMotionDetector` 임계값 조정
`MHCMotionDetector` 클래스 내 `fall_threshold` 및 `abnormal_threshold` 변수의 값을 변경하여 감지 민감도를 조정할 수 있습니다.

### `WiFiFallDetector` 낙상 시뮬레이션
`MockCSICollector`를 통해 고의적으로 낙상 상황과 유사한 CSI 데이터를 주입하여 비상 알림이 트리거되는 과정을 확인할 수 있습니다.

### `WhoFiIdentifier` 사람 식별
특정 `person_id`로 사람의 바디 시그니처를 등록한 후, 유사하거나 다른 CSI 데이터로 식별을 시도하여 모듈의 정확도를 테스트할 수 있습니다.

In [21]:
from google.colab import userdata
# userdata.get('030411') # 시크릿 관련 오류를 피하기 위해 주석 처리되었습니다.

In [22]:
# WiFiFallDetector 인스턴스 생성
fall_detector_instance = WiFiFallDetector()

print("WiFiFallDetector의 detect_fall 메서드 직접 호출:")

# 낙상 감지 메서드 호출
detection_result = fall_detector_instance.detect_fall()

print(f"최종 감지 결과: {detection_result.label}, 신뢰도: {detection_result.confidence:.2f}")

WiFiFallDetector의 detect_fall 메서드 직접 호출:
[WiFiFallDetector] result=normal, confidence=0.60
최종 감지 결과: normal, 신뢰도: 0.60


In [23]:
import random
import time

# MHCMotionDetector의 임계값:
# normal: avg <= 0.60
# abnormal_movement: 0.60 < avg <= 0.85
# fall_detected: avg > 0.85

class MockCSIReader:
    def __init__(self, scenarios: list):
        self.scenarios = scenarios
        self.current_scenario_idx = 0

    def read_csi(self) -> List[float]:
        if self.current_scenario_idx < len(self.scenarios):
            scenario_type = self.scenarios[self.current_scenario_idx]
            self.current_scenario_idx += 1

            if scenario_type == "normal":
                # 평균 0.60 이하 CSI 데이터
                return [random.uniform(0.1, 0.5) for _ in range(64)]
            elif scenario_type == "abnormal_movement":
                # 평균 0.60 초과 0.85 이하 CSI 데이터
                return [random.uniform(0.61, 0.84) for _ in range(64)]
            elif scenario_type == "fall_detected":
                # 평균 0.85 초과 CSI 데이터
                return [random.uniform(0.86, 0.99) for _ in range(64)]
            else:
                return [random.random() for _ in range(64)] # 기본값
        else:
            # 시나리오가 끝나면 계속 normal 데이터를 반환
            return [random.uniform(0.1, 0.5) for _ in range(64)]

print("WiFiSensingModule 실제 시나리오 시뮬레이션:")

# 시나리오 정의: 일반 -> 비정상 -> 낙상 -> 일반 -> 비정상
simulation_scenarios = ["normal", "abnormal_movement", "fall_detected", "normal", "abnormal_movement"]

# WiFiSensingModule 인스턴스 생성
sensing_module = WiFiSensingModule()

# MockCSIReader를 사용하여 csi_extractor 교체
sensing_module.csi_extractor = MockCSIReader(simulation_scenarios)

# 시뮬레이션 실행 (시나리오 수에 맞춰 iterations 설정)
sensing_module.run(iterations=len(simulation_scenarios), sleep_sec=0.1)

print("\nWiFiSensingModule 시뮬레이션 완료.")

WiFiSensingModule 실제 시나리오 시뮬레이션:
[WiFiSensingModule] motion_status=normal
[WiFiSensingModule] motion_status=abnormal_movement
[WiFiSensingModule] motion_status=fall_detected
[EMERGENCY] Fall detected. Alert triggered.
[WiFiSensingModule] motion_status=normal
[WiFiSensingModule] motion_status=abnormal_movement

WiFiSensingModule 시뮬레이션 완료.


In [24]:
import random

# WhoFiIdentifier 인스턴스 생성
identifier = WhoFiIdentifier()

print("WhoFiIdentifier 사람 식별 시뮬레이션:")

# 1. 사람 등록 (Enroll a person)
person_id_to_enroll = "Alice"
print(f"\n1. {person_id_to_enroll} 등록 시도...")
identifier.enroll_person(person_id_to_enroll, seconds=3) # 3초간 CSI 데이터 수집 및 등록
print(f"{person_id_to_enroll}의 바디 시그니처가 등록되었습니다.")

# 2. 등록된 사람 식별 시도 (Attempt to identify the enrolled person)
# 등록된 Alice의 시그니처와 매우 유사한 CSI 데이터 생성 (거의 동일)
# identifier.body_signature_db에 저장된 시그니처를 직접 사용합니다.
sample_csi_for_alice = identifier.body_signature_db[person_id_to_enroll][:]
# 약간의 노이즈를 추가하여 실제와 유사하게 만듭니다 (선택 사항)
sample_csi_for_alice = [max(0.0, min(1.0, s + random.uniform(-0.01, 0.01))) for s in sample_csi_for_alice]

print(f"\n2. {person_id_to_enroll} 식별 시도 (유사한 CSI 데이터)...")
identified_person_1 = identifier.identify_person(sample_csi_for_alice, threshold=0.90) # 임계값 설정
print(f"  식별 결과: {identified_person_1}")

# 3. 등록되지 않은 사람 식별 시도 (Attempt to identify an unknown person)
# 등록된 시그니처와는 다른 형태의 CSI 데이터 생성
unknown_csi = [random.uniform(0.1, 0.4) for _ in range(64)]

print("\n3. 등록되지 않은 사람 식별 시도 (다른 CSI 데이터)...")
identified_person_2 = identifier.identify_person(unknown_csi, threshold=0.90) # 동일 임계값 사용
print(f"  식별 결과: {identified_person_2}")

print("\n시뮬레이션 완료.")

WhoFiIdentifier 사람 식별 시뮬레이션:

1. Alice 등록 시도...
[WhoFi] enrolled person_id=Alice
Alice의 바디 시그니처가 등록되었습니다.

2. Alice 식별 시도 (유사한 CSI 데이터)...
[WhoFi] compare Alice: similarity=0.995
  식별 결과: Alice

3. 등록되지 않은 사람 식별 시도 (다른 CSI 데이터)...
[WhoFi] compare Alice: similarity=0.731
  식별 결과: unknown

시뮬레이션 완료.


In [25]:
import random

# WiFiFallDetector 인스턴스 생성
fall_detector = WiFiFallDetector()

print("WiFiFallDetector 낙상 감지 시뮬레이션:")

# 낙상 상황을 유도하기 위한 CSI 데이터 생성
# MHCMotionModel은 평균 진폭이 0.85보다 높으면 fall_detected, confidence=0.97을 반환합니다.
# 이를 위해 각 CSI 프레임의 평균이 높게 나오도록 데이터를 구성합니다.

# 가상의 CSICollector를 직접 생성하여, high_amplitude_csi_stream을 반환하도록 재정의합니다.
# 이는 시뮬레이션 목적이며, 실제 코드에서는 CSICollector.collect()가 호출됩니다.
class MockCSICollector:
    def collect(self, duration: int = 3) -> List[List[float]]:
        # 각 프레임의 평균이 0.85를 넘도록 데이터를 생성합니다.
        frames = max(1, duration * 5) # 예시로 15개의 프레임을 생성합니다.
        high_amplitude_csi_stream = []
        for _ in range(frames):
            # 각 프레임의 값이 0.85에서 0.99 사이가 되도록 하여 평균을 높입니다.
            high_amplitude_csi_stream.append([random.uniform(0.86, 0.99) for _ in range(64)])
        return high_amplitude_csi_stream

# 기존 fall_detector의 csi_collector를 MockCSICollector 인스턴스로 교체합니다.
fall_detector.csi_collector = MockCSICollector()

# detect_fall 메서드 호출
fall_detection_result = fall_detector.detect_fall()

print(f"\n시뮬레이션 결과: {fall_detection_result.label}, 신뢰도: {fall_detection_result.confidence:.2f}")
if fall_detection_result.label == "fall_detected" and fall_detection_result.confidence > 0.95:
    print("\n-> 낙상 감지 조건 충족 (label: fall_detected, confidence > 0.95), 비상 상황 트리거 예상")
else:
    print("\n-> 낙상 감지 조건 불충족")


WiFiFallDetector 낙상 감지 시뮬레이션:
[WiFiFallDetector] result=fall_detected, confidence=0.97
[EMERGENCY] WiFi fall detector triggered

시뮬레이션 결과: fall_detected, 신뢰도: 0.97

-> 낙상 감지 조건 충족 (label: fall_detected, confidence > 0.95), 비상 상황 트리거 예상


In [26]:
import random

# MHCMultiModalSensor 인스턴스 생성
multimodal_sensor = MHCMultiModalSensor()

print("MHCMultiModalSensor 음성 감지 기능 테스트:")

# 시나리오 1: 음성 없음 (평균 0.20 미만)
# 낮은 값의 오디오 스트림 생성
audio_no_speech = [random.uniform(0.01, 0.15) for _ in range(32)]
# 평균 계산 (실제 판단 기준)
avg_no_speech = sum(audio_no_speech) / len(audio_no_speech) if audio_no_speech else 0
print(f"\n시나리오 1: 음성 없음 (오디오 평균: {avg_no_speech:.3f})")
print(f"  음성 감지 결과: {multimodal_sensor.is_speech_present(audio_no_speech)}")

# 시나리오 2: 음성 있음 (평균 0.20 이상)
# 높은 값의 오디오 스트림 생성
audio_with_speech = [random.uniform(0.25, 0.9) for _ in range(32)]
# 평균 계산 (실제 판단 기준)
avg_with_speech = sum(audio_with_speech) / len(audio_with_speech) if audio_with_speech else 0
print(f"\n시나리오 2: 음성 있음 (오디오 평균: {avg_with_speech:.3f})")
print(f"  음성 감지 결과: {multimodal_sensor.is_speech_present(audio_with_speech)}")

# 시나리오 3: 빈 오디오 스트림
audio_empty = []
print(f"\n시나리오 3: 빈 오디오 스트림")
print(f"  음성 감지 결과: {multimodal_sensor.is_speech_present(audio_empty)}")


MHCMultiModalSensor 음성 감지 기능 테스트:

시나리오 1: 음성 없음 (오디오 평균: 0.066)
  음성 감지 결과: False

시나리오 2: 음성 있음 (오디오 평균: 0.602)
  음성 감지 결과: True

시나리오 3: 빈 오디오 스트림
  음성 감지 결과: False


In [27]:
import random

# MHCMotionDetector 인스턴스 생성 (이미 임계값이 변경된 상태)
motion_detector = MHCMotionDetector()

print("MHCMotionDetector 시뮬레이션 (새로운 임계값 적용):")

# 1. 일반 움직임 시뮬레이션 (평균이 0.60 미만인 경우)
# 0.1 ~ 0.5 범위의 랜덤 값으로 평균을 0.60 미만으로 유지
csi_normal_new = [random.uniform(0.1, 0.5) for _ in range(64)]
print(f"일반 움직임 시뮬레이션 (평균: {sum(csi_normal_new)/len(csi_normal_new):.3f})")
print(f"결과: {motion_detector.analyze(csi_normal_new)}\n")

# 2. 비정상 움직임 시뮬레이션 (평균이 0.60 이상 0.85 미만인 경우)
# 0.60 ~ 0.80 범위의 랜덤 값으로 평균을 0.60 이상 0.85 미만으로 유지
csi_abnormal_new = [random.uniform(0.60, 0.80) for _ in range(64)]
print(f"비정상 움직임 시뮬레이션 (평균: {sum(csi_abnormal_new)/len(csi_abnormal_new):.3f})")
print(f"결과: {motion_detector.analyze(csi_abnormal_new)}\n")

# 3. 낙상 감지 시뮬레이션 (평균이 0.85 이상인 경우)
# 0.85 ~ 0.98 범위의 랜덤 값으로 평균을 0.85 이상으로 유지
csi_fall_new = [random.uniform(0.85, 0.98) for _ in range(64)]
print(f"낙상 감지 시뮬레이션 (평균: {sum(csi_fall_new)/len(csi_fall_new):.3f})")
print(f"결과: {motion_detector.analyze(csi_fall_new)}\n")

MHCMotionDetector 시뮬레이션 (새로운 임계값 적용):
일반 움직임 시뮬레이션 (평균: 0.308)
결과: normal

비정상 움직임 시뮬레이션 (평균: 0.703)
결과: abnormal_movement

낙상 감지 시뮬레이션 (평균: 0.921)
결과: fall_detected



In [28]:
import random

# MHCMultiModalModel 인스턴스 생성
multimodal_model = MHCMultiModalModel()

print("MHCMultiModalModel 시뮬레이션:")

# 시나리오 1: 일반 움직임 (평균 신호 < 0.66)
# 오디오와 CSI 모두 낮은 값으로 설정
audio_normal = [random.uniform(0.1, 0.5) for _ in range(32)]
csi_normal = [random.uniform(0.1, 0.5) for _ in range(64)]
aligned_data_normal = {"audio": audio_normal, "csi": csi_normal}
predicted_normal = multimodal_model.predict(aligned_data_normal)

# 평균 신호 계산 (예측 기준이 되는 값)
signal_normal = 0.0
if audio_normal: signal_normal += sum(audio_normal) / len(audio_normal)
if csi_normal: signal_normal += sum(csi_normal) / len(csi_normal)
signal_normal /= 2 if audio_normal and csi_normal else 1

print(f"\n시나리오 1: 일반 움직임 (평균 신호: {signal_normal:.3f})")
print(f"  예측: {predicted_normal}")

# 시나리오 2: 비정상 움직임 (0.66 < 평균 신호 < 0.82)
# 오디오와 CSI 모두 중간 값으로 설정
audio_abnormal = [random.uniform(0.60, 0.75) for _ in range(32)]
csi_abnormal = [random.uniform(0.60, 0.75) for _ in range(64)]
aligned_data_abnormal = {"audio": audio_abnormal, "csi": csi_abnormal}
predicted_abnormal = multimodal_model.predict(aligned_data_abnormal)

# 평균 신호 계산
signal_abnormal = 0.0
if audio_abnormal: signal_abnormal += sum(audio_abnormal) / len(audio_abnormal)
if csi_abnormal: signal_abnormal += sum(csi_abnormal) / len(csi_abnormal)
signal_abnormal /= 2 if audio_abnormal and csi_abnormal else 1

print(f"\n시나리오 2: 비정상 움직임 (평균 신호: {signal_abnormal:.3f})")
print(f"  예측: {predicted_abnormal}")

# 시나리오 3: 낙상 감지 (평균 신호 > 0.82)
# 오디오와 CSI 모두 높은 값으로 설정
audio_fall = [random.uniform(0.80, 0.95) for _ in range(32)]
csi_fall = [random.uniform(0.80, 0.95) for _ in range(64)]
aligned_data_fall = {"audio": audio_fall, "csi": csi_fall}
predicted_fall = multimodal_model.predict(aligned_data_fall)

# 평균 신호 계산
signal_fall = 0.0
if audio_fall: signal_fall += sum(audio_fall) / len(audio_fall)
if csi_fall: signal_fall += sum(csi_fall) / len(csi_fall)
signal_fall /= 2 if audio_fall and csi_fall else 1

print(f"\n시나리오 3: 낙상 감지 (평균 신호: {signal_fall:.3f})")
print(f"  예측: {predicted_fall}")

MHCMultiModalModel 시뮬레이션:

시나리오 1: 일반 움직임 (평균 신호: 0.281)
  예측: normal

시나리오 2: 비정상 움직임 (평균 신호: 0.670)
  예측: abnormal_movement

시나리오 3: 낙상 감지 (평균 신호: 0.869)
  예측: fall_detected


In [29]:
import random

# MHCMotionModel 인스턴스 생성
motion_model = MHCMotionModel()

print("MHCMotionModel 시뮬레이션:")

# 1. 일반 움직임 시뮬레이션 (평균이 0.68 이하인 경우)
amplitude_normal = [random.uniform(0.1, 0.6) for _ in range(32)]
result_normal = motion_model.predict(amplitude_normal)
print(f"일반 움직임 시뮬레이션 (평균: {sum(amplitude_normal)/len(amplitude_normal):.3f}):")
print(f"  결과: {result_normal.label}, 신뢰도: {result_normal.confidence:.2f}\n")

# 2. 비정상 움직임 시뮬레이션 (평균이 0.68과 0.85 사이인 경우)
amplitude_abnormal = [random.uniform(0.60, 0.80) for _ in range(32)]
result_abnormal = motion_model.predict(amplitude_abnormal)
print(f"비정상 움직임 시뮬레이션 (평균: {sum(amplitude_abnormal)/len(amplitude_abnormal):.3f}):")
print(f"  결과: {result_abnormal.label}, 신뢰도: {result_abnormal.confidence:.2f}\n")

# 3. 낙상 감지 시뮬레이션 (평균이 0.85보다 높은 경우)
amplitude_fall = [random.uniform(0.80, 0.95) for _ in range(32)]
result_fall = motion_model.predict(amplitude_fall)
print(f"낙상 감지 시뮬레이션 (평균: {sum(amplitude_fall)/len(amplitude_fall):.3f}):")
print(f"  결과: {result_fall.label}, 신뢰도: {result_fall.confidence:.2f}\n")

MHCMotionModel 시뮬레이션:
일반 움직임 시뮬레이션 (평균: 0.367):
  결과: normal, 신뢰도: 0.60

비정상 움직임 시뮬레이션 (평균: 0.711):
  결과: abnormal_movement, 신뢰도: 0.83

낙상 감지 시뮬레이션 (평균: 0.880):
  결과: fall_detected, 신뢰도: 0.97



In [30]:
import random

# MHCMotionDetector 인스턴스 생성
motion_detector = MHCMotionDetector()

# 다양한 시나리오를 위한 가상의 CSI 데이터 생성
# 1. 일반 움직임 (평균이 낮은 경우)
csi_normal = [random.uniform(0.1, 0.6) for _ in range(64)]
print(f"일반 움직임 시뮬레이션 (평균: {sum(csi_normal)/len(csi_normal):.3f})")
print(f"결과: {motion_detector.analyze(csi_normal)}\n")

# 2. 비정상 움직임 (평균이 0.70과 0.92 사이인 경우)
csi_abnormal = [random.uniform(0.65, 0.85) for _ in range(64)]
print(f"비정상 움직임 시뮬레이션 (평균: {sum(csi_abnormal)/len(csi_abnormal):.3f})")
print(f"결과: {motion_detector.analyze(csi_abnormal)}\n")

# 3. 낙상 감지 (평균이 0.92보다 높은 경우)
csi_fall = [random.uniform(0.88, 0.98) for _ in range(64)]
print(f"낙상 감지 시뮬레이션 (평균: {sum(csi_fall)/len(csi_fall):.3f})")
print(f"결과: {motion_detector.analyze(csi_fall)}\n")

일반 움직임 시뮬레이션 (평균: 0.317)
결과: normal

비정상 움직임 시뮬레이션 (평균: 0.754)
결과: abnormal_movement

낙상 감지 시뮬레이션 (평균: 0.937)
결과: fall_detected

